In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DateType, IntegerType, TimestampType

#Registros - Raw Ingestion Config

In [0]:
schema_temp_ingestion = StructType([
    StructField("entity_name",          StringType(),  False),
    StructField("api_endpoint",         StringType(),  True),
    StructField("auth_method",          StringType(),  True),
    StructField("secret_name",          StringType(),  True),
    StructField("pagination_type",      StringType(),  True),
    StructField("response_format",      StringType(),  True),
    StructField("raw_path",             StringType(),  True),
    StructField("data_root_key",        StringType(),  True),
    StructField("extraction_frequency", StringType(),  True),
    StructField("is_active",            BooleanType(), True),
])

datos_ingestion = [
    ("products", "https://dummyjson.com/products", "none", "none", "offset", "json",
     "/Volumes/workspace/raw/products/", "products", "daily", True), #Products
    
    ("carts", "https://dummyjson.com/carts", "none", "none", "offset", "json",
     "/Volumes/workspace/raw/carts/", "carts", "daily", True), #Carts
    
    ("users", "https://dummyjson.com/users", "none", "none", "offset", "json",
     "/Volumes/workspace/raw/users/", "users", "daily", True), #Users
]

df_nuevas_entidades = spark.createDataFrame(datos_ingestion, schema_temp_ingestion)

# Vista temporal para que spark.sql() pueda leer este DataFrame en la celda siguiente
df_nuevas_entidades.createOrReplaceTempView("vw_nuevas_entidades")

In [0]:
spark.sql("""
    MERGE INTO workspace.control.raw_ingestion_config AS target
    USING vw_nuevas_entidades AS source
    ON target.entity_name = source.entity_name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print("Entidades registradas/actualizadas en raw_ingestion_config.")

In [0]:
spark.table("workspace.control.raw_ingestion_config").display()

#Registros - Raw Api Parameters

In [0]:
# ============================================================
# CELDA 3 — DataFrame de parámetros (PySpark puro)
# ============================================================
schema_temp_params = StructType([
    StructField("entity_name",  StringType(), False),
    StructField("param_name",   StringType(), False),
    StructField("param_value",  StringType(), True),
    StructField("param_type",   StringType(), True),
])

datos_params = [
    ("products", "limit", "100", "page_size"),
    ("products", "skip",  "0",   "offset"),
    ("carts",   "limit", "100", "page_size"),
    ("carts",   "skip",  "0",   "offset"),
    ("users",    "limit", "100", "page_size"),
    ("users",    "skip",  "0",   "offset"),
]

df_nuevos_params = spark.createDataFrame(datos_params, schema_temp_params)
df_nuevos_params.createOrReplaceTempView("vw_nuevos_params")

In [0]:
spark.sql("""
    MERGE INTO workspace.control.raw_api_parameters AS target
    USING vw_nuevos_params AS source
    ON target.entity_name = source.entity_name AND target.param_name = source.param_name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print("Parámetros registrados/actualizados en raw_api_parameters.")

In [0]:
spark.table("workspace.control.raw_api_parameters").display()

#Registros Bronze load config

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, DateType, TimestampType
)

schema_temp_bronze = StructType([
    StructField("entity_name",      StringType(),    False),
    StructField("target_table",     StringType(),    True),
    StructField("primary_key",      StringType(),    True),
    StructField("load_mode",        StringType(),    True),
    StructField("watermark_column", StringType(),    True),
    StructField("last_watermark",   TimestampType(), True),
    StructField("last_loaded_date", DateType(),      True),
    StructField("last_run_status", StringType(),    True),
    StructField("is_active",        BooleanType(),   True),
])

datos_bronze = [
    # entity_name, target_table,                 primary_key, load_mode,     watermark_column,   last_watermark, last_loaded_date, last_run_status, is_active
    ("products", "workspace.bronze.products", "id", "incremental", "meta.updatedAt", None, None, None, True),
    ("carts",    "workspace.bronze.carts",    "id", "full",        None,             None, None, None, True),
    ("users",    "workspace.bronze.users",    "id", "full",        None,             None, None, None, True),
]

df_nuevas_bronze = spark.createDataFrame(datos_bronze, schema_temp_bronze)
df_nuevas_bronze.createOrReplaceTempView("vw_nuevas_bronze")

In [0]:
spark.sql("""
    MERGE INTO workspace.control.bronze_load_config AS target
    USING vw_nuevas_bronze AS source
    ON target.entity_name = source.entity_name
    WHEN MATCHED THEN UPDATE SET
        target_table     = source.target_table,
        primary_key      = source.primary_key,
        load_mode        = source.load_mode,
        watermark_column = source.watermark_column,
        is_active        = source.is_active
    WHEN NOT MATCHED THEN INSERT *
""")

print("Entidades registradas/actualizadas en bronze_load_config.")

In [0]:
spark.table("workspace.control.bronze_load_config").display()

#Registros Silver Transform Config

In [0]:
schema_temp_silver = StructType([
    StructField("entity_name",        StringType(),  False),
    StructField("source_table",       StringType(),  True),
    StructField("target_table",       StringType(),  True),
    StructField("primary_key",        StringType(),  True),
    StructField("dedup_order_column", StringType(),  True),
    StructField("last_run_status",    StringType(),  True),
    StructField("is_active",          BooleanType(), True),
])

datos_silver = [
    ("products", "workspace.bronze.products", "workspace.silver.products", "id", "meta_updated_at",     None, True),
    ("carts",    "workspace.bronze.carts",    "workspace.silver.carts",    "id", "ingestion_timestamp", None, True),
#                                                                            ^^^^^^^^^^^^^^^^^^^^
    #                                          antes decía "_ingestion_timestamp" (con guion bajo inicial)
    ("users",    "workspace.bronze.users",    "workspace.silver.users",    "id", "ingestion_timestamp", None, True),
    #                                                                            ^^^^^^^^^^^^^^^^^^^^
    #                                          mismo ajuste necesario aquí
]

df_nuevas_silver = spark.createDataFrame(datos_silver, schema_temp_silver)
df_nuevas_silver.createOrReplaceTempView("vw_nuevas_silver")

In [0]:
spark.sql("""
    MERGE INTO workspace.control.silver_transform_config AS target
    USING vw_nuevas_silver AS source
    ON target.entity_name = source.entity_name
    WHEN MATCHED THEN UPDATE SET
        source_table       = source.source_table,
        target_table       = source.target_table,
        primary_key        = source.primary_key,
        dedup_order_column = source.dedup_order_column,
        is_active          = source.is_active
    WHEN NOT MATCHED THEN INSERT *
""")

print("Entidades registradas/actualizadas en silver_transform_config.")

In [0]:
spark.table("workspace.control.silver_transform_config").display()

#Gold

In [0]:

schema_temp_gold = StructType([
    StructField("entity_name",     StringType(),  False),
    StructField("source_table",    StringType(),  True),
    StructField("target_table",    StringType(),  True),
    StructField("last_run_status", StringType(),  True),
    StructField("is_active",       BooleanType(), True),
])

datos_gold = [
    ("dim_products",   "workspace.silver.products", "workspace.gold.dim_products",   None, True),
    ("dim_users",      "workspace.silver.users",    "workspace.gold.dim_users",      None, True),
    ("fact_cart_items","workspace.silver.carts",    "workspace.gold.fact_cart_items", None, True),
]

df_nuevas_gold = spark.createDataFrame(datos_gold, schema_temp_gold)
df_nuevas_gold.createOrReplaceTempView("vw_nuevas_gold")

In [0]:
spark.sql("""
    MERGE INTO workspace.control.gold_aggregation_config AS target
    USING vw_nuevas_gold AS source
    ON target.entity_name = source.entity_name
    WHEN MATCHED THEN UPDATE SET
        source_table = source.source_table,
        target_table = source.target_table,
        is_active    = source.is_active
    WHEN NOT MATCHED THEN INSERT *
""")

print("Entidades registradas/actualizadas en gold_aggregation_config.")

In [0]:
spark.table("workspace.control.gold_aggregation_config").display()